# Unit Tests
Run all backend tests from this notebook after each important backend change.

In [ ]:
import json
import unittest

import backend.server as server
from backend.models import create_connection_state, create_match, create_user_session
from backend.protocol import ProtocolError, decode_message, encode_message
from frontend.net_utils import decode_message as front_decode_message, encode_message as front_encode_message
import sys
import types


In [ ]:
class DummySocket:
    def __init__(self):
        self.sent = []

    def sendall(self, data):
        self.sent.append(data)

    def shutdown(self, how):
        return None

    def close(self):
        return None


def parse_sent_messages(sock):
    parsed = []
    for raw in sock.sent:
        line = raw.decode('utf-8').strip()
        parsed.append(json.loads(line))
    return parsed


def message_types(sock):
    return [message['type'] for message in parse_sent_messages(sock)]


class TestProtocol(unittest.TestCase):
    def test_encode_message_has_newline_and_shape(self):
        raw = encode_message('LOGIN', {'username': 'Ali'})
        self.assertTrue(raw.endswith(b'\n'))
        parsed = json.loads(raw.decode('utf-8'))
        self.assertEqual(parsed['type'], 'LOGIN')
        self.assertEqual(parsed['payload'], {'username': 'Ali'})

    def test_decode_message_valid(self):
        parsed = decode_message(b'{"type":"WAITING","payload":{}}')
        self.assertEqual(parsed, {'type': 'WAITING', 'payload': {}})

    def test_decode_message_rejects_invalid_messages(self):
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN"')
        with self.assertRaises(ProtocolError):
            decode_message(b'{"payload":{}}')
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN","payload":[]}')




class TestFrontendNetUtils(unittest.TestCase):
    def test_frontend_encode_message_shape(self):
        raw = front_encode_message('PING', {'x': 1})
        self.assertTrue(raw.endswith(b'\n'))
        data = json.loads(raw.decode('utf-8'))
        self.assertEqual(data['type'], 'PING')
        self.assertEqual(data['payload'], {'x': 1})

    def test_frontend_decode_message_shape(self):
        parsed = front_decode_message(b'{"type":"PONG","payload":{"ok":true}}')
        self.assertEqual(parsed['type'], 'PONG')
        self.assertEqual(parsed['payload'], {'ok': True})

    def test_frontend_decode_message_rejects_bad_payload(self):
        with self.assertRaises(ValueError):
            front_decode_message(b'{"type":"BAD","payload":[]}')
class TestServer(unittest.TestCase):
    def setUp(self):
        server.STATE = create_connection_state()
        server.RUNNING.clear()

    def register_user(self, username, port):
        sock = DummySocket()
        session = create_user_session(sock, ('127.0.0.1', port))
        server.handle_login(session, {'username': username})
        return session, sock

    def test_login_uniqueness(self):
        _, first_sock = self.register_user('Ali', 5001)
        second_sock = DummySocket()
        second_session = create_user_session(second_sock, ('127.0.0.1', 5002))
        server.handle_login(second_session, {'username': 'Ali'})

        self.assertIn('LOGIN_OK', message_types(first_sock))
        self.assertIsNone(second_session['username'])
        self.assertEqual(parse_sent_messages(second_sock)[0]['type'], 'LOGIN_REJECT')

    def test_waiting_and_watch_transitions(self):
        session, sock = self.register_user('Ali', 5001)

        server.dispatch_message(session, {'type': 'WAITING', 'payload': {}})
        self.assertIn('Ali', server.STATE['waiting_players'])

        server.dispatch_message(session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertNotIn('Ali', server.STATE['waiting_players'])
        self.assertIn('Ali', server.STATE['spectators'])

        types = message_types(sock)
        self.assertIn('WAITING', types)
        self.assertIn('WATCH_MATCH', types)

    def test_challenge_and_accept_create_match(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        maya_session, maya_sock = self.register_user('Maya', 5002)

        server.handle_challenge_player(ali_session, {'target': 'Maya'})
        self.assertEqual(server.STATE['pending_challenges'].get('Maya'), 'Ali')
        self.assertIn('CHALLENGE_PLAYER', message_types(ali_sock))
        self.assertIn('CHALLENGE_RECEIVED', message_types(maya_sock))

        server.handle_challenge_accept(maya_session, {'from': 'Ali'})
        self.assertIsNotNone(server.STATE['active_match'])
        players = set(server.STATE['active_match']['players'])
        self.assertEqual(players, {'Ali', 'Maya'})
        self.assertIn('MATCH_START', message_types(ali_sock))
        self.assertIn('MATCH_START', message_types(maya_sock))

    def test_spectator_promoted_to_player_gets_single_player_start(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        maya_session, maya_sock = self.register_user('Maya', 5002)

        server.dispatch_message(ali_session, {'type': 'WATCH_MATCH', 'payload': {}})
        server.dispatch_message(maya_session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertIn('Ali', server.STATE['spectators'])
        self.assertIn('Maya', server.STATE['spectators'])

        server.handle_challenge_player(ali_session, {'target': 'Maya'})
        server.handle_challenge_accept(maya_session, {'from': 'Ali'})

        self.assertNotIn('Ali', server.STATE['spectators'])
        self.assertNotIn('Maya', server.STATE['spectators'])

        ali_starts = [m for m in parse_sent_messages(ali_sock) if m['type'] == 'MATCH_START']
        maya_starts = [m for m in parse_sent_messages(maya_sock) if m['type'] == 'MATCH_START']
        self.assertEqual(len(ali_starts), 1)
        self.assertEqual(len(maya_starts), 1)
        self.assertFalse(ali_starts[0]['payload']['spectator'])
        self.assertFalse(maya_starts[0]['payload']['spectator'])

    def test_input_updates_pending_direction(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')
        self.assertIsNotNone(match)

        server.handle_input(ali_session, {'direction': 'UP'})
        self.assertEqual(server.STATE['active_match']['snakes']['Ali']['pending_direction'], 'UP')

        bad_sock = DummySocket()
        bad_session = create_user_session(bad_sock, ('127.0.0.1', 5010))
        bad_session['username'] = 'Ghost'
        server.handle_input(bad_session, {'direction': 'LEFT'})
        self.assertEqual(parse_sent_messages(bad_sock)[0]['type'], 'ERROR')

    def test_advance_tick_can_finish_on_timer(self):
        config = server.get_match_config()
        match = create_match(99, 'Ali', 'Maya', config)
        match['remaining_ticks'] = 1
        match['obstacles'] = []
        match['pies'] = []
        match['snakes']['Ali']['body'] = [(5, 5), (4, 5), (3, 5)]
        match['snakes']['Maya']['body'] = [(20, 5), (21, 5), (22, 5)]
        match['snakes']['Ali']['direction'] = 'RIGHT'
        match['snakes']['Ali']['pending_direction'] = 'RIGHT'
        match['snakes']['Maya']['direction'] = 'LEFT'
        match['snakes']['Maya']['pending_direction'] = 'LEFT'
        match['snakes']['Ali']['health'] = 80
        match['snakes']['Maya']['health'] = 70

        server.advance_match_one_tick(match, config)

        self.assertEqual(match['status'], 'ended')
        self.assertEqual(match['reason'], 'timer_end')
        self.assertEqual(match['winner'], 'Ali')

    def test_disconnect_marks_match_ended(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')
        self.assertEqual(match['status'], 'running')

        server.disconnect_session(ali_session)

        active = server.STATE['active_match']
        self.assertIsNotNone(active)
        self.assertEqual(active['status'], 'ended')
        self.assertEqual(active['reason'], 'player_disconnected')
        self.assertEqual(active['winner'], 'Maya')


    def test_collision_pause_flicker_and_resume_turn(self):
        config = server.get_match_config()
        match = create_match(120, 'Ali', 'Maya', config)
        match['remaining_ticks'] = 500
        match['pies'] = []
        match['obstacles'] = []

        ali = match['snakes']['Ali']
        maya = match['snakes']['Maya']

        ali['body'] = [(0, 5), (1, 5), (2, 5)]
        ali['direction'] = 'LEFT'
        ali['pending_direction'] = 'LEFT'

        maya['body'] = [(20, 10), (21, 10), (22, 10)]
        maya['direction'] = 'LEFT'
        maya['pending_direction'] = 'LEFT'

        server.advance_match_one_tick(match, config)

        self.assertEqual(ali['body'], [(0, 5), (1, 5), (2, 5)])
        self.assertEqual(ali['stun_ticks_remaining'], config['collision_pause_ticks'])
        self.assertEqual(ali['resume_direction'], 'DOWN')
        self.assertEqual(ali['health'], config['starting_health'] - config['wall_damage'])

        for _ in range(config['collision_pause_ticks'] - 1):
            before = list(ali['body'])
            server.advance_match_one_tick(match, config)
            self.assertEqual(ali['body'], before)

        self.assertEqual(ali['stun_ticks_remaining'], 1)

        server.advance_match_one_tick(match, config)
        self.assertEqual(ali['stun_ticks_remaining'], 0)
        self.assertEqual(ali['direction'], 'DOWN')

        before_move = list(ali['body'])
        server.advance_match_one_tick(match, config)
        self.assertNotEqual(ali['body'][0], before_move[0])




class TestFrontendUI(unittest.TestCase):
    def load_frontend_ui(self):
        fake_pygame = types.SimpleNamespace()
        fake_pygame.SRCALPHA = 1
        sys.modules['pygame'] = fake_pygame

        import importlib
        module = importlib.import_module('frontend.ui')
        module = importlib.reload(module)
        return module

    def test_button_state_priority(self):
        fui = self.load_frontend_ui()
        self.assertEqual(fui.get_button_state(True, True), fui.STATE_PRESSED)
        self.assertEqual(fui.get_button_state(True, False), fui.STATE_HOVER)
        self.assertEqual(fui.get_button_state(False, True), fui.STATE_IDLE)

    def test_clamp_value_bounds(self):
        fui = self.load_frontend_ui()
        self.assertEqual(fui.clamp_value(-3, 0, 10), 0)
        self.assertEqual(fui.clamp_value(13, 0, 10), 10)
        self.assertEqual(fui.clamp_value(5, 0, 10), 5)

    def test_button_click_requires_press_start_inside(self):
        fui = self.load_frontend_ui()

        class FakeRect:
            def collidepoint(self, pos):
                return pos == (1, 1)

        button = fui.UIButton.__new__(fui.UIButton)
        button.rect = FakeRect()
        button.state = fui.STATE_IDLE
        button.was_down = False
        button.pressed_inside = False

        self.assertFalse(button.update((0, 0), True))
        self.assertFalse(button.update((1, 1), False))

        self.assertFalse(button.update((1, 1), True))
        self.assertTrue(button.update((1, 1), False))


    def test_multiply_brightness_uses_additive_blend_for_brightening(self):
        fui = self.load_frontend_ui()

        class FakeOverlay:
            def __init__(self, size):
                self.size = size
                self.fill_color = None

            def fill(self, color):
                self.fill_color = color

        class FakeSurface:
            def __init__(self, size):
                self.size = size
                self.blit_calls = []

            def copy(self):
                return FakeSurface(self.size)

            def get_size(self):
                return self.size

            def blit(self, overlay, pos, special_flags=None):
                self.blit_calls.append({
                    'overlay': overlay,
                    'pos': pos,
                    'special_flags': special_flags,
                })

        def fake_surface_factory(size, _flags):
            return FakeOverlay(size)

        fui.pygame.BLEND_RGBA_ADD = 200
        fui.pygame.BLEND_RGBA_MULT = 100
        fui.pygame.Surface = fake_surface_factory

        base = FakeSurface((4, 4))
        brightened = fui.multiply_brightness(base, 1.2)

        self.assertEqual(len(brightened.blit_calls), 1)
        call = brightened.blit_calls[0]
        self.assertEqual(call['special_flags'], fui.pygame.BLEND_RGBA_ADD)
        self.assertGreater(call['overlay'].fill_color[0], 0)

class TestFrontendClientConnectionLifecycle(unittest.TestCase):
    def load_frontend_client(self):
        fake_pygame = types.SimpleNamespace()
        fake_pygame.QUIT = 256
        fake_pygame.KEYDOWN = 768
        fake_pygame.K_TAB = 9
        fake_pygame.K_RETURN = 13
        fake_pygame.K_BACKSPACE = 8
        fake_pygame.K_UP = 273
        fake_pygame.K_DOWN = 274
        fake_pygame.K_LEFT = 276
        fake_pygame.K_RIGHT = 275
        fake_pygame.K_ESCAPE = 27
        fake_pygame.K_c = ord('c')
        fake_pygame.K_a = ord('a')
        fake_pygame.K_w = ord('w')
        fake_pygame.K_v = ord('v')
        fake_pygame.K_l = ord('l')
        sys.modules['pygame'] = fake_pygame

        import importlib
        module = importlib.import_module('frontend.client')
        module = importlib.reload(module)
        return module

    def test_stale_disconnect_event_is_ignored(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['connection_id'] = 2
        state['screen'] = fclient.SCREEN_LOBBY

        state['network_queue'].put({'connection_id': 1, 'message': {'type': 'DISCONNECTED', 'payload': {}}})
        fclient.process_network_queue(state)

        self.assertEqual(state['screen'], fclient.SCREEN_LOBBY)

    def test_close_connection_clears_receive_buffer(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['recv_buffer'] = b'partial-frame'

        fclient.close_connection(state)

        self.assertEqual(state['recv_buffer'], b'')


    def test_keyboard_shortcuts_trigger_matching_button_feedback(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()

        feedback_calls = []

        def record_feedback(_state, action_name):
            feedback_calls.append((_state['screen'], action_name))

        fclient.press_button_feedback = record_feedback

        state['screen'] = fclient.SCREEN_CONNECT
        fclient.handle_connect_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_RETURN, unicode=''))

        state['screen'] = fclient.SCREEN_USERNAME
        fclient.handle_username_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_RETURN, unicode=''))

        state['screen'] = fclient.SCREEN_LOBBY
        state['self_name'] = 'Ali'
        state['online_users'] = ['Ali', 'Maya']
        state['selected_user_index'] = 0
        state['pending_challenger'] = 'Maya'

        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_c, unicode='c'))
        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_a, unicode='a'))
        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_w, unicode='w'))
        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_v, unicode='v'))

        state['screen'] = fclient.SCREEN_GAME_OVER
        fclient.handle_game_over_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_l, unicode='l'))

        expected = [
            (fclient.SCREEN_CONNECT, 'connect'),
            (fclient.SCREEN_USERNAME, 'login'),
            (fclient.SCREEN_LOBBY, 'challenge'),
            (fclient.SCREEN_LOBBY, 'accept'),
            (fclient.SCREEN_LOBBY, 'wait'),
            (fclient.SCREEN_LOBBY, 'watch'),
            (fclient.SCREEN_GAME_OVER, 'to_lobby'),
        ]
        self.assertEqual(feedback_calls, expected)


In [ ]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestProtocol))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestFrontendNetUtils))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestFrontendUI))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestFrontendClientConnectionLifecycle))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestServer))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

if not result.wasSuccessful():
    raise AssertionError('Some unit tests failed')
